# HypotheSAEs Quickstart

This notebook demonstrates basic usage of HypotheSAEs on a sample of the Yelp review dataset.
By default we use `gpt-5.2` for hypothesis generation (interpreting neurons) and `gpt-5-mini` for text annotation.

Set `OPENAI_KEY_SAE` below for OpenAI-hosted requests. If you are using a local OpenAI-compatible endpoint (for example, vLLM server mode), set `OPENAI_BASE_URL`; the key is optional for non-OpenAI hosts.


In [1]:
%load_ext autoreload
%autoreload 2

import os
os.environ['OPENAI_KEY_SAE'] = '...'  # Replace with your OpenAI API key (or set from another env var)
os.environ['OPENAI_BASE_URL'] = 'http://0.0.0.0:8001/v1'  # Optional: local OpenAI-compatible endpoint

import pandas as pd

import importlib
import hypothesaes.quickstart
importlib.reload(hypothesaes.quickstart)
from hypothesaes.quickstart import interpret_sae, interpret_sep_sae

import hypothesaes.embedding 
importlib.reload(hypothesaes.embedding)
from hypothesaes.embedding import get_local_embeddings

from hypothesaes.quickstart import train_sae

INTERPRETER_MODEL = "openai/gpt-oss-20b"
ANNOTATOR_MODEL = INTERPRETER_MODEL

# Optional request kwargs forwarded directly to the Responses API.
# Keep empty by default; set only what you need at call time.
# Example: INTERPRET_LLM_KWARGS = {"reasoning_effort": "low"}
INTERPRET_LLM_KWARGS = {"reasoning_effort": "low"}
ANNOTATION_LLM_KWARGS = {"reasoning_effort": "low"}

N_WORKERS_ANNOTATION = 10  # Lower if hitting API rate limits

**Load data**

The dataset we're using here is a subset of 20K Yelp reviews, with 2K reviews used for validation (during SAE training). 

The target variable is the `stars` column, which is a rating between 1 and 5. We treat this as a regression task.

There are also 2K reviews used for holdout evaluation, which we'll use at the end of the notebook.

In [5]:
current_dir = os.getcwd()
if current_dir.endswith("notebooks"):
    prefix = "../"
else:
    prefix = "./"
is_vast = True
if is_vast:
    train_path = os.path.join(prefix, 'data', 'vast_train.csv')
    val_path = os.path.join(prefix, 'data', 'vast_dev.csv')
    DOC_FIELD = 'post'
    TOPIC_FIELD = 'new_topic'
else:
    train_path = os.path.join(prefix, 'data', 'raw_train_all_onecol.csv')
    train_path = os.path.join(prefix, 'data', 'raw_val_all_onecol.csv')
    DOC_FIELD = "Ori Text"
    TOPIC_FIELD = "Target 1"
train_df = pd.read_csv(train_path)
val_df = pd.read_csv(val_path)
unique_train_texts  = list(train_df[DOC_FIELD].unique())
unique_train_topics = list(train_df[TOPIC_FIELD].unique())


In [6]:
val_df = val_df[~(
    val_df[DOC_FIELD].map(set(unique_train_texts).__contains__) |
    val_df[TOPIC_FIELD].map(set(unique_train_topics).__contains__)
)]
unique_val_texts  = list(val_df[DOC_FIELD].unique())
unique_val_topics = list(val_df[TOPIC_FIELD].unique())

In [8]:
embed_func = lambda texts: get_local_embeddings(texts, return_stacked=True)
train_text_embeds  = embed_func(unique_train_texts)
val_text_embeds    = embed_func(unique_val_texts)
train_topic_embeds = embed_func(unique_train_topics)
val_topic_embeds   = embed_func(unique_val_topics)

Loading weights:   0%|          | 0/134 [00:00<?, ?it/s]

Loaded model nomic-ai/modernbert-embed-base to cuda


Processing chunks:   0%|          | 0/1 [00:00<?, ?it/s]

Chunk 0:   0%|          | 0/15 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/134 [00:00<?, ?it/s]

Loaded model nomic-ai/modernbert-embed-base to cuda


... (6 outputs left out)

Loading weights:   0%|          | 0/134 [00:00<?, ?it/s]

Loaded model nomic-ai/modernbert-embed-base to cuda


Processing chunks:   0%|          | 0/1 [00:00<?, ?it/s]

Chunk 0:   0%|          | 0/4 [00:00<?, ?it/s]

In [9]:
shared_kwargs = {
    "M": 256,
    "K": 8,
    "matryoshka_prefix_lengths": [32, 256],
    "n_epochs": 300
}
text_sae = train_sae(embeddings=train_text_embeds, val_embeddings=val_text_embeds, **shared_kwargs)
topic_sae = train_sae(embeddings=train_topic_embeds, val_embeddings=val_topic_embeds, **shared_kwargs)

  0%|          | 0/300 [00:00<?, ?it/s]

Early stopping triggered after 155 epochs


  0%|          | 0/300 [00:00<?, ?it/s]

Early stopping triggered after 119 epochs


In [10]:
all_texts = list(train_df[DOC_FIELD])
all_text_embeds = embed_func(all_texts)
all_topics = list(train_df[TOPIC_FIELD])
all_topic_embeds = embed_func(all_topics)

Loading weights:   0%|          | 0/134 [00:00<?, ?it/s]

Loaded model nomic-ai/modernbert-embed-base to cuda


Processing chunks:   0%|          | 0/1 [00:00<?, ?it/s]

Chunk 0:   0%|          | 0/106 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/134 [00:00<?, ?it/s]

Loaded model nomic-ai/modernbert-embed-base to cuda


Processing chunks:   0%|          | 0/1 [00:00<?, ?it/s]

Chunk 0:   0%|          | 0/106 [00:00<?, ?it/s]

In [26]:
import numpy as np
def get_labels(df):
    return np.array(list(map(int, df['label'])))
train_labels = get_labels(train_df)
val_labels = get_labels(val_df)

In [29]:
import numpy as np
def conj_acts(texts, topics, conj=np.minimum):
    text_embeds = embed_func(texts)
    topic_embeds = embed_func(topics)
    text_acts = text_sae.get_activations(text_embeds)
    topic_acts = topic_sae.get_activations(topic_embeds)
    n_samples = text_acts.shape[0]
    conj_acts = conj(np.expand_dims(text_acts, 2), np.expand_dims(topic_acts, 1))
    flattened = np.reshape(conj_acts, (n_samples, -1))
    return flattened
df_acts = lambda df: conj_acts(df[DOC_FIELD], df[TOPIC_FIELD])
train_acts = df_acts(train_df)
val_acts   = df_acts(val_df)

Loading weights:   0%|          | 0/134 [00:00<?, ?it/s]

Loaded model nomic-ai/modernbert-embed-base to cuda


Processing chunks:   0%|          | 0/1 [00:00<?, ?it/s]

Chunk 0:   0%|          | 0/106 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/134 [00:00<?, ?it/s]

Loaded model nomic-ai/modernbert-embed-base to cuda


... (10 outputs left out)

Processing chunks:   0%|          | 0/1 [00:00<?, ?it/s]

Chunk 0:   0%|          | 0/10 [00:00<?, ?it/s]

Computing activations (batchsize=16384):   0%|          | 0/1 [00:00<?, ?it/s]

Computing activations (batchsize=16384):   0%|          | 0/1 [00:00<?, ?it/s]

In [30]:
train_acts.shape

(13477, 65536)

In [1]:
import cuml_cu13
from cuml_cu13.linear_model import LogisticRegression

ModuleNotFoundError: No module named 'cuml_cu13.linear_model'

In [ ]:
from sklearn.model_selection import PredefinedSplit, GridSearchCV
# from sklearn.linear_model import LogisticRegression

split = PredefinedSplit(test_fold=np.concat([np.full_like(train_labels, -1), np.zeros_like(val_labels)]))
param_grid = {
    "l1_ratio": [0.99],
}
cv = GridSearchCV(
    estimator=LogisticRegression(solver='saga'),
    param_grid=param_grid,
    scoring='f1_macro',
    cv=split,
)
fitted_clf = cv.fit(
    np.concat([train_acts, val_acts]),
    np.concat([train_labels, val_labels])
)

ValueError: Invalid parameter 'clf' for estimator LinearRegression(). Valid parameters are: ['copy_X', 'fit_intercept', 'n_jobs', 'positive', 'tol'].

In [ ]:
import numpy as np

In [16]:
conj_acts.shape

(13477, 256, 256)

In [8]:
topic_task_instructions = """All of the below samples are topics discussed on online fora.
Features should be more general than the topics themselves."""
# Interpret random neurons
results_topics = interpret_sae(
    texts=train_topics,
    embeddings=train_topic_embeds,
    sae=topic_sae,
    n_random_neurons=5,
    print_examples_n=3,
    task_specific_instructions=topic_task_instructions,
    interpreter_model=INTERPRETER_MODEL,
    interpret_llm_kwargs=INTERPRET_LLM_KWARGS,
)

Computing activations (batchsize=16384):   0%|          | 0/1 [00:00<?, ?it/s]

Activations shape: (3761, 256)


Generating interpretations:   0%|          | 0/5 [00:00<?, ?it/s]


Neuron 64 (0.4% active): mentions Catholic

Top activating examples:
1. saint
2. catholic sainthood
3. discovering that a woman who was a saint was not as helpful as had been expected
----------------------------------------------------------------------------------------------------

Neuron 155 (0.4% active): financial aid

Top activating examples:
1. financial aid programs
2. financial aid based on income
3. affirmative action
----------------------------------------------------------------------------------------------------

Neuron 2 (4.9% active): mentions regulation

Top activating examples:
1. regulation of business
2. regulation of companies
3. regulation of business practices
----------------------------------------------------------------------------------------------------

Neuron 217 (1.0% active): mentions bank

Top activating examples:
1. federal bailout for banks
2. federal bailouts
3. government bailout for banks
--------------------------------------------------------

In [9]:
text_task_instructions = """All of the below samples are posts from an online forum.
Features should describe a specific aspect of the post (especially what topics it discusses)."""
# Interpret random neurons
results = interpret_sae(
    texts=train_texts,
    embeddings=train_text_embeds,
    sae=text_sae,
    n_random_neurons=5,
    print_examples_n=3,
    task_specific_instructions=text_task_instructions,
    interpreter_model=INTERPRETER_MODEL,
    interpret_llm_kwargs=INTERPRET_LLM_KWARGS,
)

Computing activations (batchsize=16384):   0%|          | 0/1 [00:00<?, ?it/s]

Activations shape: (1383, 256)


Generating interpretations:   0%|          | 0/5 [00:00<?, ?it/s]


Neuron 98 (2.6% active): mentions the bar exam

Top activating examples:
1. These law schools will say and do almost anything to avoid having to reduce the number of law students, which is really the only solution to this whole mess. As an attorney who's taken and passed some of the hardest bar exams in the nation, I will tell you that bar exams are not that hard. If you can't pass them, then you don't deserve to practice. Law schools need to quit whining about the bar and take a long look in the mirror. Either find better students, take fewer of them, or tank your ranking. Either way, blaming the system is just excuse-making.
2. The basis for this problem is that law school is a big money maker for the universities. Unlike medical school or grad school in the sciences and engineering, there are no expensive labs to maintain. Just a big lecture hall and an instructor. That has led to a proliferation of law schools and a decline in the average ability of those accepted to law school. M